In [4]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [5]:
# Initialize openAI

MODEL = "llama3.2"
openai = OpenAI(base_url='http://127.0.0.1:11434/v1', api_key='ollama')

In [8]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com#wp--skip-link--target',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://www.linkedin.com/in/eddonner/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2025/05/28/connecting-my-courses-become-an-llm-expert-and-leader/',
 'https://twitter.com/edwarddonner',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/proficient/',
 'https://news.ycombinator.com',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://www.facebook.com/edward.donner.52',
 'https://nebula.io/?utm_source=ed&utm_medium=re

In [22]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.

You MUST respond strictly with valid JSON using the following structure:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [23]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [11]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/
https://edwarddonner.com/outsmart/
https://edwarddonner.com#wp--skip-link--target
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://www.linkedin.com/in/eddonner/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/avatar/
https://edwarddonner.com/posts/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2025/05/28/connecting-my-courses-become-an-llm-expert-and-leader/
https://twitter.com/edwarddonner
https://edwarddonner.com/connect-four/
https://e

In [24]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [25]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling llama3.2
Found 3 relevant links


{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'Company/Profile',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'LinkedIn Profile',
   'url': 'https://www.linkedin.com/in/eddonner/'}]}

In [26]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama3.2
Found 5 relevant links


{'links': [{'type': 'about page', 'url': 'https://huggingface.co'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'support', 'url': 'https://huggingface.co/support'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'changelog', 'url': 'https://huggingface.co/changelog'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [27]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    
    for link in relevant_links.get('links', []):
        # Check if the model returned a dict or just a string
        if isinstance(link, dict):
            link_type = link.get('type', 'relevant page')
            link_url = link.get('url', '')
        else:
            # Fallback if Ollama returned a string instead of a dict
            link_type = "relevant page"
            link_url = str(link)
            
        if link_url:
            result += f"\n\n### Link: {link_type}\n"
            result += fetch_website_contents(link_url)
            
    return result

In [28]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company but remember they are coming from a low-educational backgroud so you have to make sure the words you use should include more example from daily lives and use as simple terms and phrases as possible for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [29]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [30]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama3.2
Found 9 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nHardware\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\ndeepseek-ai/DeepSeek-V4-Flash-0731\nUpdated\n4 days ago\n•\n433k\n•\n2.34k\nMiniMaxAI/MiniMax-H3\nUpdated\nabout 16 

In [31]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [32]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama3.2
Found 3 relevant links


# Welcome to Hugging Face

## About Us

Hugging Face is a community-driven platform that brings together the machine learning community to collaborate on models, datasets, and applications. Our mission is to make AI accessible to everyone, and we're making it happen.

## What We Do

We offer a range of tools and services to help you get started with artificial intelligence:

*   ** Models**: Browse our vast library of pre-trained models, from natural language processing to computer vision, and more.
*   **Datasets**: Access over 500,000 datasets for AI model training, including datasets specifically designed for image recognition and natural language processing.
*   **Spaces**: Run your AI experiments in dedicated environments with our robust infrastructure.
*   **Buckets**: Store and manage your dataset data securely.

## Our Community

Our community is made up of passionate individuals from all walks of life who share a common goal: to harness the power of artificial intelligence for good. Whether you're a researcher, developer, or entrepreneur, we have a space for you:

*   **Join the conversation**: Connect with our active community and stay updated on the latest developments in AI technology.
*   **Collaborate with others**: Work together with like-minded individuals to tackle real-world challenges.

## Careers & Opportunities

Want to join the Hugging Face team?

*   **Internships**: Great for students who want hands-on experience with AI development.
*   **Full-time roles**: Pursue a career that combines machine learning and technology, from software engineer to project manager.
*   **Innovation partners**: Discover opportunities for collaboration and growth by partnering with us on cutting-edge projects.

If you're up-to-date on the latest developments in AI and would like to be part of an exciting new generation of research – we urge you to get in touch! Our customer support team would be delighted to hear from you.

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [33]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [34]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama3.2
Found 10 relevant links


# Welcome to Hugging Face: Building a Better Future through AI Collaboration

Hugging Face is a platform that unites the machine learning community in developing tomorrow's intelligent systems today. Our mission is to make high-quality pre-trained models, datasets, and applications accessible to everyone.

## Join the Movement

Whether you're an researcher, entrepreneur, or student, we invite you to be part of our vibrant community. Our platform features:

* **2M+ Pre-Trained Models**: Explore our vast library of models and tools for building AI-driven applications.
* **500k+ Datasets**: Access a vast repository of labeled datasets, making it easier to develop accurate machine learning models.

## Collaborative Spaces

Hugging Face provides an open-source platform that encourages collaboration. Our various spaces include:

* **Agents**: A hub for developing agent-based architectures and reinforcement learning techniques.
* **Datasets**: Browse our collection of high-quality labeled datasets, made by experts in their fields.
* **Models**: Explore our extensive library of pre-trained models, optimized for a variety of tasks.

## Meet the Talent

Our team is composed of passionate individuals who share your enthusiasm for building a better future through AI collaboration. We're always looking for talented minds to join our mission! Check out our [Careers page](https://hf.co/careers) to learn more about current and upcoming job opportunities.

At Hugging Face, we celebrate diversity and inclusion. Our [GitHub repository](https://github.com/huggingface) is open-source, making it possible for anyone to contribute and improve our codebase.

Stay up-to-date with the latest news from Hugging Face by following us on Twitter: [@huggingface](https://twitter.com/huggingface).

In [35]:
stream_brochure("Canva", "https://www.canva.com/")

Selecting relevant links for https://www.canva.com/ by calling llama3.2
Found 8 relevant links


Brochure of Canva


Canva is a company that makes design tools simple and accessible. Like having a superpower tool that lets you create amazing things.


## Our Purpose

At Canva, we want to help everyone create amazing visual content – whether it's a small business owner or an educator. We believe in empowering people with the power of good design.


## Our Services


*   Design tools: Create awesome designs fast and easy.
*   Photo editor: Edit photos like a pro for better results.
*   Video editor: Cut, trim,  resize videos quickly for YouTube, vlogs, etc 
*   Brand management: Protect your brand with templates that you can own
*   AI features: Canva's Artificial Intelligence is revolutionizing how we design and make our stuff
*   Templates Marketplace &  App store where a vast array of templates, graphics are available to be used.